In [1]:
import numpy as np
import scipy.stats as sps

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from functools import lru_cache
from Levenshtein import distance as levenshtein_distance
from collections import defaultdict
import numpy as np

! pip install python-Levenshtein

# Прочтение таблиц

In [48]:
df = pd.read_excel("King_Charles.xlsx")
df.head()

,Rank,Read.count,Read.proportion,CDR3.nucleotide.sequence,CDR3.amino.acid.sequence,bestVGene,bestJGene
0,213,225,0.000100,TGTGCCAGCAGTTTTCGCCGAGAGATGAACACTGAAGCTTTCTTT,CASSFRREMNTEAFF,TRBV12-4,TRBJ1-1
1,214,225,0.000100,TGTGCCAGCTCCCAGGGGGCAGTCGGGGAGCTGTTTTTT,CASSQGAVGELFF,TRBV2,TRBJ2-2
2,212,226,0.000100,TGTGCCAGCAGTTTGTGGACCTACAATGAGCAGTTCTTC,CASSLWTYNEQFF,TRBV12-4,TRBJ2-1
3,211,226,0.000100,TGTGCCAGCATTTTGACAGGGGCGAACACCGGGGAGCTGTTTTTT,CASILTGANTGELFF,TRBV2,TRBJ2-2
4,207,227,0.000101,TGTGCCAGCAGCCTTTACGGGGGGACAGGGCTACATGATGAGCAGT...,CASSLYGGTGLHDEQFF,TRBV7-6,TRBJ2-1


In [6]:
database = pd.read_excel("vdjdb.slim.xlsx")

In [50]:
database_exp = database.rename(columns={'cdr3': 'CDR3.amino.acid.sequence'})

In [52]:
df = df.sort_values(by='Read.count', ascending=False)
df

,Rank,Read.count,Read.proportion,CDR3.nucleotide.sequence,CDR3.amino.acid.sequence,bestVGene,bestJGene
214,0,5435,2.416853e-03,TGCGCCAGCAGCTTGGGAGGGGATACGCAGTATTTT,CASSLGGDTQYF,TRBV5-1,TRBJ2-3
213,1,4969,2.209631e-03,TGCGCCAGCAGGGTGGGACTAGCGGGAGGGCCTGTAGATGAGCAGT...,CASRVGLAGGPVDEQFF,TRBV5-1,TRBJ2-1
212,2,4400,1.956606e-03,TGTGCCAGCAGCTCCTATGAATCCCCCTACAATGAGCAGTTCTTC,CASSSYESPYNEQFF,TRBV11-2,TRBJ2-1
211,3,2794,1.242445e-03,TGCGCCAGCAGCTTGGAGGGGACAGACTATGGCTACACCTTC,CASSLEGTDYGYTF,TRBV5-1,TRBJ1-2
210,4,2650,1.178410e-03,TGTGCCAGCAGCTTTTTGTCCAGTGAAGCTTTCTTT,CASSFLSSEAFF,TRBV7-2,TRBJ1-1
...,...,...,...,...,...,...,...
431750,906994,1,4.446832e-07,TGTGCCAGCACCTACACGGGAGTGGCCTACGAGCAGTACTTC,CASTYTGVAYEQYF,TRBV12-4,TRBJ2-7
431751,1089087,1,4.446832e-07,TGTGCCAGCACCTACACGGGGGGGAACATTCAGTACTTC,CASTYTGGNIQYF,TRBV6-1,TRBJ2-4
431752,687953,1,4.446832e-07,TGTGCCAGCACCTACACTAGCGGGGCTGGCGGGGAGCTGTTTTTT,CASTYTSGAGGELFF,TRBV2,TRBJ2-2
431753,529877,1,4.446832e-07,TGTGCCAGCACCTACACTAGCGGGGGGACCACAGATACGCAGTATTTT,CASTYTSGGTTDTQYF,TRBV12-3,TRBJ2-3


In [110]:
data_claster = df[:10000]
data_claster

,Rank,Read.count,Read.proportion,CDR3.nucleotide.sequence,CDR3.amino.acid.sequence,bestVGene,bestJGene
214,0,5435,0.002417,TGCGCCAGCAGCTTGGGAGGGGATACGCAGTATTTT,CASSLGGDTQYF,TRBV5-1,TRBJ2-3
213,1,4969,0.002210,TGCGCCAGCAGGGTGGGACTAGCGGGAGGGCCTGTAGATGAGCAGT...,CASRVGLAGGPVDEQFF,TRBV5-1,TRBJ2-1
212,2,4400,0.001957,TGTGCCAGCAGCTCCTATGAATCCCCCTACAATGAGCAGTTCTTC,CASSSYESPYNEQFF,TRBV11-2,TRBJ2-1
211,3,2794,0.001242,TGCGCCAGCAGCTTGGAGGGGACAGACTATGGCTACACCTTC,CASSLEGTDYGYTF,TRBV5-1,TRBJ1-2
210,4,2650,0.001178,TGTGCCAGCAGCTTTTTGTCCAGTGAAGCTTTCTTT,CASSFLSSEAFF,TRBV7-2,TRBJ1-1
...,...,...,...,...,...,...,...
122828,9524,10,0.000004,TGTGCCAGCAGCTTAAACAGGGTTACGAGCACTGAAGCTTTCTTT,CASSLNRVTSTEAFF,TRBV7-2,TRBJ1-1
122821,9803,10,0.000004,TGTGCCAGCAGCTCTGGACACCCGGCTGAAAAACTGTTTTTT,CASSSGHPAEKLFF,TRBV7-2,TRBJ1-4
122822,9102,10,0.000004,TGTGCCAGCAGCTCTGGGTTGTACAGGGGTGACGGCAATCAGCCCC...,CASSSGLYRGDGNQPQHF,TRBV11-1,TRBJ1-5
122823,10257,10,0.000004,TGTGCCAGCAGCTCTTACAATGAGCAGTTCTTC,CASSSYNEQFF,TRBV7-8,TRBJ2-1


# Без кластеризации (Наивный подход)


In [195]:
full = pd.merge(df, database_exp , on='CDR3.amino.acid.sequence', how='inner')

In [203]:
full = full[['CDR3.amino.acid.sequence', 'Read.proportion', 'bestVGene',	'bestJGene', 'v.segm',	'j.segm', 'mhc.class', 'species', 'antigen.species', 'antigen.gene']]
full

,CDR3.amino.acid.sequence,Read.proportion,bestVGene,bestJGene,v.segm,j.segm,mhc.class,species,antigen.species,antigen.gene
0,CASSLGGDTQYF,2.416853e-03,TRBV5-1,TRBJ2-3,TRBV28*01,TRBJ2-1*01,MHCI,HomoSapiens,HIV-1,Pol
1,CASSLSTDTQYF,2.321246e-04,TRBV7-6,TRBJ2-3,TRBV12-3*01,TRBJ2-3*01,MHCI,HomoSapiens,SARS-CoV-2,ORF1ab
2,CASSLSTDTQYF,2.321246e-04,TRBV7-6,TRBJ2-3,TRBV28*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,pp65
3,CASSLSTDTQYF,2.321246e-04,TRBV7-6,TRBJ2-3,TRBV5-4*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE1
4,CASSLSTDTQYF,2.321246e-04,TRBV7-6,TRBJ2-3,TRBV5-4*01,TRBJ2-3*01,MHCI,HomoSapiens,EBV,EBNA3A
...,...,...,...,...,...,...,...,...,...,...
23392,CASTVSSYNEQFF,4.446832e-07,TRBV14,TRBJ2-1,TRBV6-6*01,TRBJ2-1*01,MHCI,HomoSapiens,SARS-CoV-2,Nucleocapsid
23393,CASTGAGELFF,4.446832e-07,TRBV7-9,TRBJ2-2,TRBV12-3*01,TRBJ2-2*01,MHCI,HomoSapiens,InfluenzaA,M
23394,CASTGGNEQFF,4.446832e-07,TRBV11-2,TRBJ2-1,TRBV25-1*01,TRBJ2-1*01,MHCI,HomoSapiens,InfluenzaA,NP
23395,CASTGTAYGYTF,4.446832e-07,TRBV25-1,TRBJ1-2,TRBV19*01,TRBJ1-2*01,MHCI,HomoSapiens,HIV-1,Gag


In [205]:
full_vj = full.loc[(full['bestVGene'].str[:7] == full['v.segm'].str[:7]) & (full['bestJGene'].str[:7] == full['j.segm'].str[:7])]
full_vj

,CDR3.amino.acid.sequence,Read.proportion,bestVGene,bestJGene,v.segm,j.segm,mhc.class,species,antigen.species,antigen.gene
11,CSVGTGGTNEKLFF,1.107261e-04,TRBV29-1,TRBJ1-4,TRBV29-1*01,TRBJ1-4*01,MHCI,HomoSapiens,EBV,BMLF1
12,CSVGTGGTNEKLFF,1.107261e-04,TRBV29-1,TRBJ1-4,TRBV29-1*01,TRBJ1-4*01,MHCII,HomoSapiens,InfluenzaA,NP
15,CASSLEGDQPQHF,8.982600e-05,TRBV5-1,TRBJ1-5,TRBV5-1*01,TRBJ1-5*01,MHCI,HomoSapiens,CMV,IE1
16,CASSLEGDQPQHF,8.982600e-05,TRBV5-1,TRBJ1-5,TRBV5-1*01,TRBJ1-5*01,MHCI,HomoSapiens,InfluenzaA,NP
17,CASSLEGDQPQHF,8.982600e-05,TRBV5-1,TRBJ1-5,TRBV5-1*01,TRBJ1-5*01,MHCI,HomoSapiens,SARS-CoV-2,Spike
...,...,...,...,...,...,...,...,...,...,...
23314,CASNRGNTGELFF,4.446832e-07,TRBV12-3,TRBJ2-2,TRBV12-3*01,TRBJ2-2*01,MHCI,HomoSapiens,CMV,IE1
23328,CASTWGNQPQHF,4.446832e-07,TRBV12-4,TRBJ1-5,TRBV12-4*01,TRBJ1-5*01,MHCII,HomoSapiens,InfluenzaA,M1
23333,CASTRGGETQYF,4.446832e-07,TRBV12-3,TRBJ2-5,TRBV12-3*01,TRBJ2-5*01,MHCI,HomoSapiens,CMV,pp65
23339,CASTLGNTEAFF,4.446832e-07,TRBV12-3,TRBJ1-1,TRBV12-4*01,TRBJ1-1*01,MHCI,HomoSapiens,InfluenzaA,M


In [207]:
full_vj['antigen.species'].value_counts()

antigen.species
CMV                     917
EBV                     260
SARS-CoV-2              229
InfluenzaA              209
HomoSapiens             145
HIV-1                    94
DENV                     52
HCV                      47
YFV                      26
Mtb                      12
TriticumAestivum         11
Influenza B              10
PlasmodiumFalciparum      6
LCMV                      4
AdV                       3
HTLV-1                    3
MCMV                      2
HCoV-HKU1                 1
Trypanosoma cruzi         1
SIV                       1
Name: count, dtype: int64

# Функция кластеризации

In [149]:
# Кэшированная функция для вычисления расстояния Левенштейна
@lru_cache(maxsize=None)
def cached_levenshtein_distance(seq1, seq2):
    return levenshtein_distance(seq1, seq2)

# Функция для кластеризации последовательностей
def cluster_sequences(sequences, max_distance=1):
    clusters = []
    for seq in sequences:
        added_to_cluster = False
        for cluster in clusters:
            if any(cached_levenshtein_distance(seq, existing_seq) <= max_distance for existing_seq in cluster):
                cluster.append(seq)
                added_to_cluster = True
                break
        if not added_to_cluster:
            clusters.append([seq])
    return clusters

# Функция для объединения кластеров
def merge_clusters(clusters, max_distance=1):
    merged_clusters = []
    for cluster in clusters:
        added_to_cluster = False
        for merged_cluster in merged_clusters:
            if any(cached_levenshtein_distance(seq, existing_seq) <= max_distance for seq in cluster for existing_seq in merged_cluster):
                merged_cluster.extend(cluster)
                added_to_cluster = True
                break
        if not added_to_cluster:
            merged_clusters.append(cluster)
    return merged_clusters

# Пример последовательностей аминокислот
sequences = data_claster['CDR3.amino.acid.sequence']

# Разделение данных на части
chunk_size = len(sequences) // 100
chunks = [sequences[i:i + chunk_size] for i in range(0, len(sequences), chunk_size)]

# Кластеризация каждой части
all_clusters = []
for chunk in chunks:
    all_clusters.extend(cluster_sequences(chunk, max_distance=2))

# Объединение кластеров
final_clusters = merge_clusters(all_clusters, max_distance=2)

# Вывод результатов
# Создание DataFrame
cluster_data = []
for i, cluster in enumerate(final_clusters):
    for seq in cluster:
        cluster_data.append({'Cluster': i + 1, 'Sequence': seq})

results_df = pd.DataFrame(cluster_data)

In [151]:
# Создание нового DataFrame с данными из data_claster и номером кластера
cluster_assignments = {}
for i, cluster in enumerate(final_clusters):
  for seq in cluster:
    cluster_assignments[seq] = i + 1

results_df = data_claster.copy()
results_df['Cluster'] = results_df['CDR3.amino.acid.sequence'].map(cluster_assignments)

# Вывод нового DataFrame results_df
results_df


,Rank,Read.count,Read.proportion,CDR3.nucleotide.sequence,CDR3.amino.acid.sequence,bestVGene,bestJGene,Cluster
214,0,5435,0.002417,TGCGCCAGCAGCTTGGGAGGGGATACGCAGTATTTT,CASSLGGDTQYF,TRBV5-1,TRBJ2-3,1
213,1,4969,0.002210,TGCGCCAGCAGGGTGGGACTAGCGGGAGGGCCTGTAGATGAGCAGT...,CASRVGLAGGPVDEQFF,TRBV5-1,TRBJ2-1,2
212,2,4400,0.001957,TGTGCCAGCAGCTCCTATGAATCCCCCTACAATGAGCAGTTCTTC,CASSSYESPYNEQFF,TRBV11-2,TRBJ2-1,3
211,3,2794,0.001242,TGCGCCAGCAGCTTGGAGGGGACAGACTATGGCTACACCTTC,CASSLEGTDYGYTF,TRBV5-1,TRBJ1-2,4
210,4,2650,0.001178,TGTGCCAGCAGCTTTTTGTCCAGTGAAGCTTTCTTT,CASSFLSSEAFF,TRBV7-2,TRBJ1-1,5
...,...,...,...,...,...,...,...,...
122828,9524,10,0.000004,TGTGCCAGCAGCTTAAACAGGGTTACGAGCACTGAAGCTTTCTTT,CASSLNRVTSTEAFF,TRBV7-2,TRBJ1-1,5871
122821,9803,10,0.000004,TGTGCCAGCAGCTCTGGACACCCGGCTGAAAAACTGTTTTTT,CASSSGHPAEKLFF,TRBV7-2,TRBJ1-4,5872
122822,9102,10,0.000004,TGTGCCAGCAGCTCTGGGTTGTACAGGGGTGACGGCAATCAGCCCC...,CASSSGLYRGDGNQPQHF,TRBV11-1,TRBJ1-5,5873
122823,10257,10,0.000004,TGTGCCAGCAGCTCTTACAATGAGCAGTTCTTC,CASSSYNEQFF,TRBV7-8,TRBJ2-1,1


# Подсчёт Total Read Proportion

In [154]:

# Подсчет количества последовательностей в каждом кластере
cluster_counts = {i + 1: len(cluster) for i, cluster in enumerate(final_clusters)}
# Подсчет суммарного Read.proportion для каждого кластера
cluster_proportions = {}
for index, row in data_claster.iterrows():
  seq = row['CDR3.amino.acid.sequence']
  proportion = row['Read.proportion']
  for i, cluster in enumerate(final_clusters):
    if seq in cluster:
      cluster_proportions[i + 1] = cluster_proportions.get(i + 1, 0) + proportion
      break

# Создание DataFrame с суммарными пропорциями, номерами кластеров и количеством последовательностей
proportions_data = []
for cluster_number, total_proportion in cluster_proportions.items():
  proportions_data.append({'Cluster': cluster_number, 
               'Total Read Proportion': total_proportion, 
               'Sequence Count': cluster_counts[cluster_number]})



# Сортировка по Sequence Count

In [157]:
proportions_df = pd.DataFrame(proportions_data)
proportions_df = proportions_df.sort_values(by='Sequence Count', ascending=False)
proportions_df

,Cluster,Total Read Proportion,Sequence Count
0,1,0.018716,1775
57,61,0.001600,159
193,28,0.002045,154
16,17,0.002226,151
3,4,0.002169,134
...,...,...,...
2228,2232,0.000012,1
2227,2231,0.000012,1
2226,2230,0.000012,1
2224,2228,0.000012,1


# Сортировка по Total Read Proportion

In [160]:
proportions_df_sorted = proportions_df.sort_values(by='Total Read Proportion', ascending=False)
#proportions_df_sorted = proportions_df_sorted.loc[proportions_df_sorted['Sequence Count'] > 1]
proportions_df_sorted

,Cluster,Total Read Proportion,Sequence Count
0,1,0.018716,1775
16,17,0.002226,151
1,2,0.002210,1
3,4,0.002169,134
193,28,0.002045,154
...,...,...,...
5497,5498,0.000004,1
5502,5503,0.000004,1
5496,5497,0.000004,1
5503,5504,0.000004,1


# Выбор кластера 

In [163]:
full = pd.merge(results_df, database_exp , on='CDR3.amino.acid.sequence', how='inner')

In [165]:
full = full[['gene', 'Cluster', 'CDR3.amino.acid.sequence', 'Read.proportion', 'bestVGene',	'bestJGene', 'v.segm',	'j.segm', 'mhc.class', 'species', 'antigen.species', 'antigen.gene']]
full

,gene,Cluster,CDR3.amino.acid.sequence,Read.proportion,bestVGene,bestJGene,v.segm,j.segm,mhc.class,species,antigen.species,antigen.gene
0,TRB,1,CASSLGGDTQYF,0.002417,TRBV5-1,TRBJ2-3,TRBV28*01,TRBJ2-1*01,MHCI,HomoSapiens,HIV-1,Pol
1,TRB,1,CASSLSTDTQYF,0.000232,TRBV7-6,TRBJ2-3,TRBV12-3*01,TRBJ2-3*01,MHCI,HomoSapiens,SARS-CoV-2,ORF1ab
2,TRB,1,CASSLSTDTQYF,0.000232,TRBV7-6,TRBJ2-3,TRBV28*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,pp65
3,TRB,1,CASSLSTDTQYF,0.000232,TRBV7-6,TRBJ2-3,TRBV5-4*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE1
4,TRB,1,CASSLSTDTQYF,0.000232,TRBV7-6,TRBJ2-3,TRBV5-4*01,TRBJ2-3*01,MHCI,HomoSapiens,EBV,EBNA3A
...,...,...,...,...,...,...,...,...,...,...,...,...
643,TRB,17,CASSFGGNTEAFF,0.000004,TRBV12-3,TRBJ1-1,TRBV12-3*01,TRBJ1-1*01,MHCI,HomoSapiens,HomoSapiens,MLANA
644,TRB,17,CASSFGGNTEAFF,0.000004,TRBV12-3,TRBJ1-1,TRBV27*01,TRBJ1-1*01,MHCI,HomoSapiens,CMV,pp65
645,TRB,1,CASSLLAGAADTQYF,0.000004,TRBV6-4,TRBJ2-3,TRBV7-9*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE2
646,TRB,48,CASSLAGGTGELFF,0.000004,TRBV12-3,TRBJ2-2,TRBV5-1*01,TRBJ2-2*01,MHCI,HomoSapiens,CMV,IE1


In [169]:
full_vj = full.loc[(full['bestVGene'].str[:7] == full['v.segm'].str[:7]) & (full['bestJGene'].str[:7] == full['j.segm'].str[:7]) & (full['gene'] == 'TRB')]
full_vj

,gene,Cluster,CDR3.amino.acid.sequence,Read.proportion,bestVGene,bestJGene,v.segm,j.segm,mhc.class,species,antigen.species,antigen.gene
11,TRB,176,CSVGTGGTNEKLFF,0.000111,TRBV29-1,TRBJ1-4,TRBV29-1*01,TRBJ1-4*01,MHCI,HomoSapiens,EBV,BMLF1
12,TRB,176,CSVGTGGTNEKLFF,0.000111,TRBV29-1,TRBJ1-4,TRBV29-1*01,TRBJ1-4*01,MHCII,HomoSapiens,InfluenzaA,NP
15,TRB,220,CASSLEGDQPQHF,0.000090,TRBV5-1,TRBJ1-5,TRBV5-1*01,TRBJ1-5*01,MHCI,HomoSapiens,CMV,IE1
16,TRB,220,CASSLEGDQPQHF,0.000090,TRBV5-1,TRBJ1-5,TRBV5-1*01,TRBJ1-5*01,MHCI,HomoSapiens,InfluenzaA,NP
17,TRB,220,CASSLEGDQPQHF,0.000090,TRBV5-1,TRBJ1-5,TRBV5-1*01,TRBJ1-5*01,MHCI,HomoSapiens,SARS-CoV-2,Spike
...,...,...,...,...,...,...,...,...,...,...,...,...
612,TRB,1,CASSLGNTEAFF,0.000004,TRBV5-1,TRBJ1-1,"TRBV5-1*01,TRBV11-3*01,TRBV6-6*01",TRBJ1-1*01,MHCI,HomoSapiens,CMV,IE1
622,TRB,75,CASSSSGSSYNEQFF,0.000004,TRBV7-9,TRBJ2-1,TRBV7-9*01,TRBJ2-1*01,MHCI,HomoSapiens,Influenza B,NP
634,TRB,1,CASSFGQQETQYF,0.000004,TRBV12-4,TRBJ2-5,TRBV12-4*01,TRBJ2-5*01,MHCI,HomoSapiens,EBV,BMLF1
638,TRB,1,CASSLGTDTQYF,0.000004,TRBV12-3,TRBJ2-3,TRBV12-3*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,pp65


In [171]:
full_total_prop = pd.merge(proportions_df_sorted, full_vj , on='Cluster', how='inner')
full_total_prop 

,Cluster,Total Read Proportion,Sequence Count,gene,CDR3.amino.acid.sequence,Read.proportion,bestVGene,bestJGene,v.segm,j.segm,mhc.class,species,antigen.species,antigen.gene
0,1,0.018716,1775,TRB,CASSFTDTQYF,0.000019,TRBV12-4,TRBJ2-3,TRBV12-4*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE1
1,1,0.018716,1775,TRB,CASSLGTGSYEQYF,0.000018,TRBV12-4,TRBJ2-7,"TRBV12-4*01,TRBV5-4*01",TRBJ2-7*01,MHCI,HomoSapiens,CMV,IE1
2,1,0.018716,1775,TRB,CASSLEETQYF,0.000017,TRBV5-1,TRBJ2-5,TRBV5-1*01,TRBJ2-5*01,MHCII,HomoSapiens,CMV,pp65
3,1,0.018716,1775,TRB,CASSQETQYF,0.000012,TRBV3-1,TRBJ2-5,TRBV3-1*01,TRBJ2-5*01,MHCI,HomoSapiens,CMV,IE1
4,1,0.018716,1775,TRB,CASSFQETQYF,0.000010,TRBV5-6,TRBJ2-5,TRBV5-6*01,TRBJ2-5*01,MHCI,HomoSapiens,CMV,pp65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2296,0.000021,2,TRB,CSARDNYEQYF,0.000011,TRBV20-1,TRBJ2-7,TRBV20-1*01,TRBJ2-7*01,MHCI,HomoSapiens,HIV-1,Gag
76,2296,0.000021,2,TRB,CSARDNYEQYF,0.000010,TRBV20-1,TRBJ2-7,TRBV20-1*01,TRBJ2-7*01,MHCI,HomoSapiens,HIV-1,Gag
77,1715,0.000015,1,TRB,CSARDPSGTSNEQFF,0.000015,TRBV20-1,TRBJ2-1,TRBV20-1*01,TRBJ2-1*01,MHCI,HomoSapiens,CMV,pp65
78,1917,0.000011,2,TRB,CASSLQGAGNTIYF,0.000013,TRBV12-4,TRBJ1-3,TRBV12-3*01,TRBJ1-3*01,MHCI,HomoSapiens,HomoSapiens,PMEL


In [173]:
full_vj['Cluster'].value_counts()

Cluster
1       29
220     12
48       5
176      4
4        4
17       3
1449     2
1772     2
2296     2
1917     2
75       2
27       1
1715     1
876      1
885      1
496      1
432      1
453      1
434      1
28       1
61       1
12       1
321      1
1671     1
Name: count, dtype: int64

# Обработка для одного выбранного кластера

In [185]:
results_df_one_cluster = results_df.loc[results_df['Cluster'] == 1]
results_df_one_cluster = results_df_one_cluster.rename(columns={'Sequence': 'CDR3.amino.acid.sequence'})
results_df_one_cluster

,Rank,Read.count,Read.proportion,CDR3.nucleotide.sequence,CDR3.amino.acid.sequence,bestVGene,bestJGene,Cluster
214,0,5435,0.002417,TGCGCCAGCAGCTTGGGAGGGGATACGCAGTATTTT,CASSLGGDTQYF,TRBV5-1,TRBJ2-3,1
196,18,1368,0.000608,TGTGCCAGCAGTCCAAGCACAGATACGCAGTATTTT,CASSPSTDTQYF,TRBV27,TRBJ2-3,1
165,49,652,0.000290,TGTGCCAGCAGTAACACAGATACGCAGTATTTT,CASSNTDTQYF,TRBV28,TRBJ2-3,1
145,70,522,0.000232,TGTGCCAGCAGCTTAAGCACAGATACGCAGTATTTT,CASSLSTDTQYF,TRBV7-6,TRBJ2-3,1
61,153,287,0.000128,TGTGCCAGCAGCCTCGGAGGCTACGAGCAGTACTTC,CASSLGGYEQYF,TRBV27,TRBJ2-7,1
...,...,...,...,...,...,...,...,...
123237,9711,10,0.000004,TGTGCCAGCGGCCTTGTGGGGGCATACAATGAGCAGTTCTTC,CASGLVGAYNEQFF,TRBV12-4,TRBJ2-1,1
123030,9267,10,0.000004,TGTGCCAGCAGTGAATCGCAGGGCAGCTCCTACAATGAGCAGTTCTTC,CASSESQGSSYNEQFF,TRBV25-1,TRBJ2-1,1
123186,9429,10,0.000004,TGTGCCAGCAGTTTCGTTTCGGGTTACACAGATACGCAGTATTTT,CASSFVSGYTDTQYF,TRBV12-4,TRBJ2-3,1
123028,10251,10,0.000004,TGTGCCAGCAGTGAATACAATGAGCAGTTCTTC,CASSEYNEQFF,TRBV2,TRBJ2-1,1


In [187]:
full = pd.merge(results_df_one_cluster, database_exp , on='CDR3.amino.acid.sequence', how='inner')

In [189]:
full = full[['gene','Cluster', 'CDR3.amino.acid.sequence', 'Read.proportion', 'bestVGene',	'bestJGene', 'v.segm',	'j.segm', 'mhc.class', 'species', 'antigen.species', 'antigen.gene']]
full

,gene,Cluster,CDR3.amino.acid.sequence,Read.proportion,bestVGene,bestJGene,v.segm,j.segm,mhc.class,species,antigen.species,antigen.gene
0,TRB,1,CASSLGGDTQYF,0.002417,TRBV5-1,TRBJ2-3,TRBV28*01,TRBJ2-1*01,MHCI,HomoSapiens,HIV-1,Pol
1,TRB,1,CASSLSTDTQYF,0.000232,TRBV7-6,TRBJ2-3,TRBV12-3*01,TRBJ2-3*01,MHCI,HomoSapiens,SARS-CoV-2,ORF1ab
2,TRB,1,CASSLSTDTQYF,0.000232,TRBV7-6,TRBJ2-3,TRBV28*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,pp65
3,TRB,1,CASSLSTDTQYF,0.000232,TRBV7-6,TRBJ2-3,TRBV5-4*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE1
4,TRB,1,CASSLSTDTQYF,0.000232,TRBV7-6,TRBJ2-3,TRBV5-4*01,TRBJ2-3*01,MHCI,HomoSapiens,EBV,EBNA3A
...,...,...,...,...,...,...,...,...,...,...,...,...
431,TRB,1,CASSLGTDTQYF,0.000004,TRBV12-3,TRBJ2-3,TRBV7-9*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE1
432,TRB,1,CASSLGTDTQYF,0.000004,TRBV12-3,TRBJ2-3,TRBV12-3*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,pp65
433,TRB,1,CASSFSYEQYF,0.000004,TRBV12-3,TRBJ2-7,TRBV6-3*01,TRBJ2-7*01,MHCI,HomoSapiens,EBV,EBNA4
434,TRB,1,CASSLLAGAADTQYF,0.000004,TRBV6-4,TRBJ2-3,TRBV7-9*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE2


# Разборки с V и J

In [192]:
full_vj = full.loc[(full['bestVGene'].str[:7] == full['v.segm'].str[:7]) & (full['bestJGene'].str[:7] == full['j.segm'].str[:7])& (full['gene'] == 'TRB')]
full_vj

,gene,Cluster,CDR3.amino.acid.sequence,Read.proportion,bestVGene,bestJGene,v.segm,j.segm,mhc.class,species,antigen.species,antigen.gene
23,TRB,1,CASSFTDTQYF,0.000019,TRBV12-4,TRBJ2-3,TRBV12-4*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE1
26,TRB,1,CASSLGTGSYEQYF,0.000018,TRBV12-4,TRBJ2-7,"TRBV12-4*01,TRBV5-4*01",TRBJ2-7*01,MHCI,HomoSapiens,CMV,IE1
28,TRB,1,CASSLEETQYF,0.000017,TRBV5-1,TRBJ2-5,TRBV5-1*01,TRBJ2-5*01,MHCII,HomoSapiens,CMV,pp65
54,TRB,1,CASSQETQYF,0.000012,TRBV3-1,TRBJ2-5,TRBV3-1*01,TRBJ2-5*01,MHCI,HomoSapiens,CMV,IE1
89,TRB,1,CASSFQETQYF,0.000010,TRBV5-6,TRBJ2-5,TRBV5-6*01,TRBJ2-5*01,MHCI,HomoSapiens,CMV,pp65
91,TRB,1,CASSLAYEQYF,0.000010,TRBV7-2,TRBJ2-7,TRBV7-2*01,TRBJ2-7*01,MHCI,HomoSapiens,InfluenzaA,M
118,TRB,1,CASSLEGYEQYF,0.000009,TRBV12-4,TRBJ2-7,TRBV12-4*01,TRBJ2-7*01,MHCI,HomoSapiens,CMV,IE1
195,TRB,1,CASRSSGNTIYF,0.000007,TRBV12-4,TRBJ1-3,TRBV12-5*01,TRBJ1-3*01,MHCI,HomoSapiens,CMV,IE1
196,TRB,1,CASSPRGQGNTGELFF,0.000007,TRBV5-1,TRBJ2-2,TRBV5-1*01,TRBJ2-2*01,MHCII,HomoSapiens,SARS-CoV-2,Spike
197,TRB,1,CASSPGQGTDTQYF,0.000007,TRBV5-1,TRBJ2-3,TRBV5-1*01,TRBJ2-3*01,MHCI,HomoSapiens,CMV,IE2
